# Exp8 side-branch: is `relabel_new`'s translator cost a circuit fact or a learning-dynamics artifact?

**Scope.** This is a side-branch of Exp8, running in parallel with the main d=12 capacity-window
notebook (`Exp8_v8_valid_capacity.ipynb`) — don't merge the two, don't edit that notebook from here.

**Question.** `relabel_new` (remap 4 terminal ids to 4 brand-new ids) converges at 72k steps to
heldout_ce ≈ 0.045, while `relabel_perm` (permute the same 4 ids among themselves) converges to
≈ 0.0016 — a ~28x gap, despite both being a single one-hop embedding-table lookup with zero
reordering (tau = displacement = 0 for both). Either (a) `a(g)` is picking up a real circuit cost,
or (b) it's contaminated by SGD needing to learn a fresh embedding representation from a cold
start — in which case the whole translator-based `a(g)` metric is suspect for every rendering
that involves novel/cold-start parameters, not just this pair.

**Already ruled out (see handoff, not reproduced here — these don't need re-running):**
- Not a stale-AUC artifact: the gap holds at the final checkpoint's `heldout_ce` directly.
- Not a frequency confound: every target id (4,5,6,7) occurs exactly as often as its pre-relabel
  original (0,1,2,3) across 5000 sampled trees — confirmed by direct count.
- Probably not pure training noise: the gap persists even at each curve's best transient point
  across 10 checkpoints (~25x), though this is only 10-point resolution, so "leans toward," not
  "proves."

**What this notebook does, in order:**
1. Check whether the seed-1 replicate (`relabel_new`, d=16, 72k steps) has finished, and what it shows.
2. **Main task** — hand-construct exact weights for `exp8_lib.py`'s `TinyRotaryGPT` that solve
   `relabel_new` with zero training, to show the circuit is trivial regardless of SGD learning cost.
3. Secondary/confirmatory test (only if 2 is ambiguous): frozen-embedding retrain.

**Before cell 2 (construction) can be written for real**, this needs `TinyRotaryGPT`'s actual
definition — embedding table, block structure, whether embedding/unembedding are tied, and the
output head. Run the "fetch architecture" cell below on `sentinel` (where `exp8_lib.py` lives) and
read the printed source before filling in the construction cells — they're scaffolded with TODOs
tied to specific things that output will answer.

## 1. Seed-1 replicate — is the gap robust to init seed?

Rules out "this one run got unlucky." If seed 1 also lands near `heldout_ce` ≈ 0.04, the gap is robust to init seed. If it lands near ≈ 0.002, the finding may be a fluke and sections 2–3 below are moot — stop and report that instead.

In [4]:
import os, json

ROOT_LONG = "./exp8_long"  # same root as the main long-run d=16 translators

p = f"{ROOT_LONG}/translator/relabel_new_d16_s1.json"
if os.path.exists(p):
    j = json.load(open(p))
    print("seed 1 heldout_ce:", j["heldout_ce"], "| heldout_acc:", j["heldout_acc"])
    # compare directly against the seed-0 numbers already on record
    j0 = json.load(open(f"{ROOT_LONG}/translator/relabel_new_d16_s0.json"))
    j0p = json.load(open(f"{ROOT_LONG}/translator/relabel_perm_d16_s0.json"))
    print("seed 0 relabel_new  heldout_ce:", j0["heldout_ce"])
    print("seed 0 relabel_perm heldout_ce:", j0p["heldout_ce"])
    if j["heldout_ce"] > 0.02:
        print("\n=> gap replicates at seed 1 (not an unlucky init). Proceed to section 2.")
    elif j["heldout_ce"] < 0.005:
        print("\n=> seed 1 lands near relabel_perm's floor. The seed-0 finding may be a fluke.")
        print("   Stop here and report this before doing anything in section 2.")
    else:
        print("\n=> ambiguous — neither clearly near 0.04 nor near 0.002. Inspect both curves directly.")
else:
    print("not done yet. tail of the run log:")
    print(os.popen(f"tail -n 5 {ROOT_LONG}/logs/tr_relabel_new_d16_s1.txt").read())

not done yet. tail of the run log:
step   5000 out-CE 0.4184
step  10000 out-CE 0.2340
step  15000 out-CE 0.8210



## 2. The real test: construct the circuit by hand (no training)

**Claim to test:** `relabel_new` is, in the theory's own terms, a constant, position-independent,
content-independent relabeling — copy the input token, then at the unembedding step route id `k`'s
hidden state to the output logit for id `k+4` instead of `k` (for k in {0,1,2,3}; identity for
every other id). If a human can write down weights that do exactly this with ~100% exact-match
accuracy, that proves by construction — not inference — that the ~28x cost gap is a learning-dynamics
artifact, not a circuit-cost fact. Sections 3 (frozen-embedding retrain) and 4 of the handoff become
unnecessary if this succeeds cleanly.

### 2a. Fetch the architecture first

Run this on `sentinel` and read the output before touching anything below it. This is the "first
message" step the handoff calls for — don't skip it and guess.

In [2]:
import inspect
import exp8_lib as X

print("VOCAB:", X.VOCAB)
try:
    print("BOS:", X.BOS)
except Exception as e:
    print("BOS lookup failed:", e)

print("\n--- TinyRotaryGPT source ---")
print(inspect.getsource(X.TinyRotaryGPT))

VOCAB: 13
BOS: {'A': 9, 'A2': 10, 'G': 11}

--- TinyRotaryGPT source ---
    class TinyRotaryGPT(nn.Module):
        def __init__(self, vocab_size, n_embd, n_layer=12, n_head=4, max_seq_len=512):
            super().__init__(); assert n_embd % n_head == 0 and (n_embd // n_head) % 2 == 0, "head_dim must be even for RoPE"
            self.tok_emb = nn.Embedding(vocab_size, n_embd); self.blocks = nn.ModuleList([Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
            self.ln_f = nn.LayerNorm(n_embd); self.head = nn.Linear(n_embd, vocab_size, bias=False)
        def forward(self, idx, targets=None):
            x = self.tok_emb(idx)
            for blk in self.blocks: x = blk(x)
            logits = self.head(self.ln_f(x)); loss = None
            if targets is not None: loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            return logits, loss



In [3]:
import inspect
print(inspect.getsource(X.train_translator))
# also helpful: whatever builds the (input, target) pairs for a given rendering, if it's a
# separate function (e.g. X.make_translator_batch / X.render_pair / similar) — check
# train_translator's body for what it calls and print that too if it's not inlined.

    def train_translator(g, seed, d, steps, root, device, n_head=4, n_layer=12, pool_n=120_000, eval_n=1000, batch=32, lr=3e-3, win=500, row_len=320):
        rpath = tr_paths(root, g, d, seed)
        if os.path.exists(rpath): print("done already:", rpath); return
        dev = torch.device(device); rules, _ = build_grammar(GRAMMAR_SEED); spec = TRANSFORMS[g]; ds = seed if seed != 0 else 1
        rows, ostart, rlen = make_translator_rows(rules, spec, pool_n, 5000 + ds, row_len=row_len)
        R = torch.from_numpy(rows).to(dev); OS = torch.from_numpy(ostart).to(dev); RL = torch.from_numpy(rlen).to(dev)
        torch.manual_seed(seed); model = TinyRotaryGPT(VOCAB, d, n_layer, n_head, row_len + 1).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, fused=(dev.type == "cuda")); ar = torch.arange(row_len - 1, device=dev)
        import contextlib
        ac = (lambda: torch.autocast(device_type="cuda", dtype=torch.bfloat16)) if dev.type == "cuda" else None
        def mask

In [5]:
import inspect
print(inspect.getsource(X.make_translator_rows))

def make_translator_rows(grammar, spec, n_trees, seed, max_leaves=120, row_len=320):
    """row = [BOS_G, g(z)..., SEP, canonical(z)...]; ostart = index of the first canonical token. Loss only on canonical tokens."""
    rng = random.Random(seed); rows = np.zeros((n_trees, row_len), np.int16); ostart = np.zeros(n_trees, np.int64); rlen = np.zeros(n_trees, np.int64); k = 0
    while k < n_trees:
        t = sample_tree(grammar, rng); c = render(t, {})
        if len(c) > max_leaves: continue
        g = render(t, spec); row = [BOS["G"]] + g + [SEP] + c; assert len(row) <= row_len, len(row)
        rows[k, : len(row)] = row; ostart[k] = len(g) + 2; rlen[k] = len(row); k += 1
    return rows, ostart, rlen



In [7]:
import torch, torch.nn.functional as F, numpy as np, os, json, contextlib

def _masked(ar, os_, rl):
    return ((ar[None, :] + 1 >= os_[:, None]) & (ar[None, :] + 1 < rl[:, None])).float()

def train_translator_custom(g, seed, d, steps, root, device, n_head=4, n_layer=12,
                             pool_n=120_000, eval_n=1000, batch=32, lr=3e-3, win=500, row_len=320,
                             init_embed_from=None, tag=""):
    """Standalone copy of exp8_lib.train_translator's loop (verbatim logic), extended with an
    optional embedding warm-start. Never touches exp8_lib.py or its result files.

    init_embed_from: (src_model, term_map) where term_map is {target_id: source_id}. Before
        training starts, model.tok_emb.weight[target_id] := src_model.tok_emb.weight[source_id].
        Everything else (data, optimizer, loop) is identical to the stock function.

    Writes to root/translator/{g}_d{d}_s{seed}{tag}.json -- `tag` keeps this from ever colliding
    with or overwriting the stock results.
    """
    dev = torch.device(device)
    rules, _ = X.build_grammar(X.GRAMMAR_SEED)
    spec = X.TRANSFORMS[g]
    ds = seed if seed != 0 else 1
    rows, ostart, rlen = X.make_translator_rows(rules, spec, pool_n, 5000 + ds, row_len=row_len)
    R = torch.from_numpy(rows).to(dev); OS = torch.from_numpy(ostart).to(dev); RL = torch.from_numpy(rlen).to(dev)

    torch.manual_seed(seed)
    model = X.TinyRotaryGPT(X.VOCAB, d, n_layer, n_head, row_len + 1).to(dev)

    if init_embed_from is not None:
        src_model, term_map = init_embed_from
        with torch.no_grad():
            for tgt_id, src_id in term_map.items():
                model.tok_emb.weight.data[tgt_id] = src_model.tok_emb.weight.data[src_id].clone().to(dev)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, fused=(dev.type == "cuda"))
    ar = torch.arange(row_len - 1, device=dev)
    ac = (lambda: torch.autocast(device_type="cuda", dtype=torch.bfloat16)) if dev.type == "cuda" else None

    curve = []; acc = 0.0; cnt = 0
    for step in range(steps):
        idx = torch.randint(0, R.shape[0], (batch,), device=dev); b = R[idx].long(); x, y = b[:, :-1], b[:, 1:]
        with (ac() if ac else contextlib.nullcontext()): logits, _ = model(x)
        ce = F.cross_entropy(logits.reshape(-1, logits.shape[-1]).float(), y.reshape(-1), reduction="none").view(batch, -1)
        m = _masked(ar, OS[idx], RL[idx]); loss = (ce * m).sum() / m.sum()
        opt.zero_grad(); loss.backward(); opt.step(); acc += loss.item(); cnt += 1
        if cnt == win:
            curve.append(acc / cnt); acc = 0.0; cnt = 0
            if len(curve) % 10 == 0: print(f"step {step + 1:>6d} out-CE {curve[-1]:.4f}", flush=True)

    erows, eos, erl = X.make_translator_rows(rules, spec, eval_n, 777777, row_len=row_len)
    model.eval(); tot = 0.0; tok = 0; hit = 0
    with torch.no_grad():
        for i in range(0, eval_n, 64):
            b = torch.from_numpy(erows[i:i + 64].astype(np.int64)).to(dev); x, y = b[:, :-1], b[:, 1:]
            logits, _ = model(x)
            ce = F.cross_entropy(logits.reshape(-1, logits.shape[-1]).float(), y.reshape(-1), reduction="none").view(b.shape[0], -1)
            m = _masked(ar, torch.from_numpy(eos[i:i + 64]).to(dev), torch.from_numpy(erl[i:i + 64]).to(dev))
            tot += (ce * m).sum().item(); tok += m.sum().item(); hit += ((logits.argmax(-1) == y).float() * m).sum().item()

    res = dict(g=g, d=d, seed=seed, steps=steps, curve=curve, auc=float(np.mean(curve)),
               final_train=float(np.mean(curve[-4:])), heldout_ce=tot / tok, heldout_acc=hit / tok,
               win=win, init_embed_from=("frozen" if init_embed_from else "random"))
    out_path = f"{root}/translator/{g}_d{d}_s{seed}{tag}.json"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    json.dump(res, open(out_path, "w"))
    print(json.dumps({k: v for k, v in res.items() if k != "curve"}), flush=True)
    return model, res

In [ ]:
ROOT_LONG = "./exp8_long"
DEVICE    = "cuda:2"   # check nvidia-smi before assuming this is free
D, STEPS, SEED = 16, 72000, 0
TERM_MAP  = {4: 0, 5: 1, 6: 2, 7: 3}   # target_id -> source_id (reversed from the adapter-cost TERM_MAP, which is source->target)

SMOKE_TEST = False   # run a cheap few-thousand-step check first; flip to False for the real 72k run

if SMOKE_TEST:
    steps = 2000
    print("--- smoke test only (2000 steps, not a real result) ---")
else:
    steps = STEPS

# Step A: reference model with already-meaningful embeddings for ids 0-3 (identity rendering:
# canonical ids appear in both the prefix and the loss target, maximally reinforced).
ref_model, ref_res = train_translator_custom("identity", SEED, D, steps, ROOT_LONG, DEVICE,
                                              tag="_ref_for_frozen_init" + ("_smoke" if SMOKE_TEST else ""))
print("reference (identity) heldout_ce:", ref_res["heldout_ce"])

# Step B: relabel_new, warm-started from the reference model's embeddings for ids 0-3.
frozen_model, frozen_res = train_translator_custom("relabel_new", SEED, D, steps, ROOT_LONG, DEVICE,
                                                     init_embed_from=(ref_model, TERM_MAP),
                                                     tag="_frozen_init" + ("_smoke" if SMOKE_TEST else ""))
print("relabel_new (frozen init) heldout_ce:", frozen_res["heldout_ce"])

# compare against the existing stock numbers already on disk
j0  = json.load(open(f"{ROOT_LONG}/translator/relabel_new_d16_s0.json"))
j0p = json.load(open(f"{ROOT_LONG}/translator/relabel_perm_d16_s0.json"))
print("\n--- comparison ---")
print("relabel_new  (random init, stock, 72k):", j0["heldout_ce"])
print("relabel_new  (frozen init, this run)  :", frozen_res["heldout_ce"], "| steps used:", steps)
print("relabel_perm (random init, stock, 72k):", j0p["heldout_ce"])

In [10]:
ROOT_LONG = "./exp8_long"
DEVICE    = "cuda:2"
D, SEED = 16, 0
REF_STEPS, TARGET_STEPS = 24000, 72000   # asymmetric on purpose, see reasoning above
TERM_MAP  = {4: 0, 5: 1, 6: 2, 7: 3}

# Step A: reference (identity) — should saturate well before 72k; sanity-check it rather than assume.
ref_model, ref_res = train_translator_custom("identity", SEED, D, REF_STEPS, ROOT_LONG, DEVICE,
                                              tag="_ref_for_frozen_init_24k")
print("reference (identity) heldout_ce:", ref_res["heldout_ce"], "| heldout_acc:", ref_res["heldout_acc"])
if ref_res["heldout_acc"] < 0.995:
    print("WARNING: reference hasn't clearly saturated at 24k (expected ~0.9998 at 72k) — "
          "the warm-start embeddings may not be 'already meaningful' yet. Consider bumping REF_STEPS before trusting Step B.")

# Step B: relabel_new, warm-started, full 72k to stay comparable to the original 0.045 / 0.0617 numbers.
frozen_model, frozen_res = train_translator_custom("relabel_new", SEED, D, TARGET_STEPS, ROOT_LONG, DEVICE,
                                                     init_embed_from=(ref_model, TERM_MAP),
                                                     tag="_frozen_init")
print("relabel_new (frozen init) heldout_ce:", frozen_res["heldout_ce"])

j0  = json.load(open(f"{ROOT_LONG}/translator/relabel_new_d16_s0.json"))
j0p = json.load(open(f"{ROOT_LONG}/translator/relabel_perm_d16_s0.json"))
print("\n--- comparison (all at 72k target steps) ---")
print("relabel_new  (random init, stock)  :", j0["heldout_ce"])
print("relabel_new  (frozen init, this run):", frozen_res["heldout_ce"])
print("relabel_perm (random init, stock)  :", j0p["heldout_ce"])

step   5000 out-CE 0.0381
step  10000 out-CE 0.0088
step  15000 out-CE 0.0083
step  20000 out-CE 0.0326
{"g": "identity", "d": 16, "seed": 0, "steps": 24000, "auc": 0.08523912568741406, "final_train": 0.004091351229122665, "heldout_ce": 0.0007586501783923657, "heldout_acc": 0.9997337687693018, "win": 500, "init_embed_from": "random"}
reference (identity) heldout_ce: 0.0007586501783923657 | heldout_acc: 0.9997337687693018
step   5000 out-CE 0.1011
step  10000 out-CE 0.1573
step  15000 out-CE 0.4916
step  20000 out-CE 0.0494
step  25000 out-CE 0.0521
step  30000 out-CE 0.0189
step  35000 out-CE 0.0505
step  40000 out-CE 0.0049
step  45000 out-CE 0.0042
step  50000 out-CE 0.0080
step  55000 out-CE 0.0152
step  60000 out-CE 0.0133
step  65000 out-CE 0.0133
step  70000 out-CE 0.0016
{"g": "relabel_new", "d": 16, "seed": 0, "steps": 72000, "auc": 0.10346479949922974, "final_train": 0.005303178416390438, "heldout_ce": 0.0018558547052558985, "heldout_acc": 0.9994409144155337, "win": 500, "init

### 2b. What the fetched source confirms (and the one thing still open)

Confirmed from `TinyRotaryGPT`:
- **Head is NOT tied** — `self.head = nn.Linear(n_embd, vocab_size, bias=False)` is a separate
  parameter from `self.tok_emb`. This simplifies the construction a lot: the embedding table and
  the unembedding can be set independently, no tied-weight workaround needed.
- Standard pre-block embed → N blocks → `ln_f` (plain `nn.LayerNorm`) → `head`. No tied weights,
  no extra residual-destroying ops outside the blocks.
- `VOCAB = 13`, and BOS ids `{A: 9, A2: 10, G: 11}` sit above the relabel range (0-3 source, 4-7
  `relabel_new` targets), so they're untouched by this construction either way.

**Generic no-op trick (works without needing `Block`'s internals):** instead of hunting for the
exact attention-output-projection / MLP-down-projection attribute names inside `Block`, zero the
weight *and* bias of every `nn.Linear` found anywhere inside each block. Any `x = x + sublayer(x)`
residual structure then reduces to `x = x` regardless of what `sublayer` actually contains
internally (attention's OV circuit, MLP's two linears, whatever RoPE does to Q/K — all of it routes
through at least one `nn.Linear`, and a zero-weight, zero-bias linear outputs zero for any input,
and the usual activations map 0→0). This avoids needing `Block`'s source at all.

**The one thing still open, and it matters more than the architecture did:** what does
`train_translator` actually feed the model — is this an autoregressive copy/reconstruction of the
*rendered* `relabel_new` string itself (ids 4-7 appear in the input, the model should reproduce
them), or a seq2seq translation from the *canonical* (language A, ids 0-3) rendering to the
`relabel_new` rendering (ids 0-3 in, ids 4-7 out)? The construction's unembedding rows need to be
set differently depending on which it is (see `mode` in the construction cell below) — read
`train_translator`'s source (fetched in the cell right after 2a) before finalizing.

### 2c. Construction

In [ ]:
import torch, torch.nn as nn

TERM_MAP = {0: 4, 1: 5, 2: 6, 3: 7}  # relabel_new's actual mapping (confirm against X.TRANSFORMS["relabel_new"])

def zero_out_blocks(model):
    """Generic no-op trick: zero every nn.Linear's weight and bias inside every block.
    Doesn't need Block's internal attribute names — works for any attn/MLP implementation that
    routes its output through nn.Linear layers (true for essentially any standard transformer
    block, RoPE included, since RoPE only rotates Q/K before the linear projections)."""
    with torch.no_grad():
        for block in model.blocks:
            for m in block.modules():
                if isinstance(m, nn.Linear):
                    m.weight.zero_()
                    if m.bias is not None:
                        m.bias.zero_()
    return model


def construct_relabel_weights(model, term_map=TERM_MAP, mode="copy"):
    """Hand-construct weights that solve relabel_new with zero training.

    mode="copy": the task is autoregressive copy/reconstruction of the relabel_new-rendered string
        itself (ids 4-7 appear in the INPUT; model should reproduce them in the output).
    mode="translate": the task is canonical (ids 0-3) IN -> relabel_new rendering (ids 4-7) OUT.

    TODO: set `mode` once train_translator's source (fetched above) confirms which one this is —
    the tie-breaking logic below differs between the two and a wrong choice will silently produce
    a tied argmax (see notes above this cell).
    """
    zero_out_blocks(model)
    n_embd = model.tok_emb.weight.shape[1]
    vocab = X.VOCAB
    assert n_embd >= vocab, f"n_embd={n_embd} < vocab={vocab}: can't use orthogonal one-hot rows at this width (true for d=16, NOT for d=12 — see note below)"

    with torch.no_grad():
        # hand-design an orthogonal embedding table: row v = scaled standard basis vector e_v.
        # LN(c * e_v) is a permutation-symmetric function of v (same (active_value, inactive_value)
        # pair for every v, just at a different index), so dot products with another one-hot-based
        # row are maximized exactly at the matching index -- a provably correct, tie-free argmax
        # readout, which is the whole point of doing this by construction rather than guessing at
        # whatever embeddings training would have found.
        E = torch.zeros(vocab, n_embd)
        c = 10.0
        for v in range(vocab):
            E[v, v] = c
        model.tok_emb.weight.data[:vocab] = E

        head_w = torch.zeros(vocab, n_embd)
        mapped_targets = set(term_map.values())
        if mode == "copy":
            # input already contains ids 4-7 (and every other id as itself) -> identity readout
            # everywhere, INCLUDING on the relabel targets themselves.
            for v in range(vocab):
                head_w[v] = E[v]
        elif mode == "translate":
            # input contains ids 0-3 (and every other id as itself) -> ids 0-3 must route to 4-7,
            # and critically ids 0-3's OWN head rows must not also win (that would tie with
            # whichever id elsewhere legitimately maps to itself) -- so the source ids' own head
            # rows are never a correct target for any input under this task; zero them so they
            # can never accidentally tie.
            for k, g in term_map.items():
                head_w[g] = E[k]
            for v in range(vocab):
                if v not in term_map and v not in mapped_targets:
                    head_w[v] = E[v]
            # head_w[k] for k in term_map stays zero -- never a correct target, never wins ties
        else:
            raise ValueError(mode)
        model.head.weight.data = head_w
    return model

print("fill in mode=\"copy\" or \"translate\" once train_translator's source settles which one this is")

### 2d. Evaluate: does the hand-built model solve `relabel_new` with ~100% exact match?

Reuses the same tree-sampling / rendering machinery `exp8_lib` already exposes (`X.sample_tree`,
`X.render`, `X.TRANSFORMS`), so this doesn't need anything new from the model class beyond a forward
pass.

In [ ]:
import random

def eval_construction(model, g="relabel_new", mode="copy", n_trees=500, seed=777, device="cpu"):
    rules, sbl = X.build_grammar(X.GRAMMAR_SEED)   # TODO: confirm this matches train_translator's own data prep
    rng = random.Random(seed)
    model = model.to(device).eval()
    n_correct, n_total = 0, 0
    with torch.no_grad():
        for _ in range(n_trees):
            t = X.sample_tree(rules, rng)
            if mode == "copy":
                src = tgt = X.render(t, X.TRANSFORMS[g])
            elif mode == "translate":
                src = X.render(t, {})                 # canonical / language-A rendering
                tgt = X.render(t, X.TRANSFORMS[g])     # relabel_new rendering
            else:
                raise ValueError(mode)
            x = torch.tensor([src], device=device)
            logits, _ = model(x)                       # TODO: confirm this matches train_translator's forward call
            pred = logits.argmax(-1).squeeze(0).tolist()
            n_correct += sum(p == t_ for p, t_ in zip(pred, tgt))
            n_total += len(tgt)
    acc = n_correct / n_total
    print(f"{g} ({mode}): exact-match token accuracy = {acc:.4f}  ({n_correct}/{n_total})")
    return acc

# model = X.TinyRotaryGPT(vocab_size=X.VOCAB, n_embd=16, n_layer=..., n_head=...)   # TODO: n_layer/n_head from the d16-long run's actual CLI args
# model = construct_relabel_weights(model, mode="copy")   # or "translate" -- set once confirmed
# eval_construction(model, "relabel_new", mode="copy")
#
# then the comparison construction for relabel_perm (handoff step 4): same recipe, different term_map,
# confirms identical complexity for a wildly different training cost:
# model_perm = X.TinyRotaryGPT(vocab_size=X.VOCAB, n_embd=16, n_layer=..., n_head=...)
# model_perm = construct_relabel_weights(model_perm, term_map=<relabel_perm's real term_map from X.TRANSFORMS["relabel_perm"]>, mode="copy")
# eval_construction(model_perm, "relabel_perm", mode="copy")
print("uncomment above once mode and n_layer/n_head are confirmed")

**Reading the result:**
- ~100% exact match on `relabel_new` (and an equally simple construction for `relabel_perm`) → the
  ~28x `a(g)` gap is **definitely** a learning-dynamics artifact, not a circuit-cost fact. No need
  for section 3 except as a secondary confirmation. This is a finding about the `a(g)` metric itself
  — worth flagging to [[anwoy]], clearly labeled exploratory (n=1 rendering pair, found by inspecting
  an anomaly, not predicted in advance), kept separate from the main d=12 capacity-window result.
- Construction fails or is ambiguous (e.g., tied embedding/unembedding makes a clean relabel-only
  circuit awkward) → that's itself informative: the architecture's parameter-sharing choices may
  matter to the cost story. Note it and move to section 3.

## 3. Secondary/confirmatory test: frozen-embedding retrain

Only needed if section 2 leaves ambiguity, or as a second line of evidence alongside a clean
construction result.

Initialize `relabel_new`'s target embeddings (rows for ids 4,5,6,7) as **copies** of the
already-meaningful embeddings for ids 0,1,2,3 (not random init), then train normally with
everything else unchanged.
- Gap mostly closes → confirms the problem is learning a fresh representation from a cold start,
  consistent with section 2's construction argument.
- Gap barely moves even with this head start → something else is going on; re-examine whether
  section 2's construction is actually reachable by this architecture under SGD (a circuit existing
  in principle isn't always easy for gradient descent to find — a separate, known phenomenon — but
  that shouldn't be conflated with "the circuit doesn't exist").

**This needs a surgical edit to the shared `exp8_lib.py` on `sentinel`**
(`/home/anwoy/exp8_lib.py`, used by ~50 people). Get the file's `TinyRotaryGPT` class and
`train_translator` function (roughly lines 150–240 per earlier tracebacks) before proposing a diff.
Prefer a new CLI flag or a standalone variant script over editing `TinyRotaryGPT` itself if avoidable.

In [ ]:
print(inspect.getsource(X.train_translator))  # read this before proposing any diff to exp8_lib.py

## Notes

- Working style: terse, direct, technically precise; honest pushback over validation; diagnose
  before fixing; self-contained notebook cells, not standalone `.py` scripts; conclusions from
  existing data over new compute proposed after results are in.
- Compute: `sentinel` (shared, 50 users, GPU 2 is hers, `/home/anwoy` home dir, env at
  `~/envs/interp/`), or `optimus` (H200). Check `nvidia-smi` before assuming a device is free.
- This notebook is self-contained and separate from `Exp8_v8_valid_capacity.ipynb` (the main
  d=12 capacity-window track) — don't merge them.